# Automatic Linspace Estimation for Reweighting

When running `parallel_reweight`, you need to provide a `lnspace` array — the range
of artificial EIR offset values to scan.  Choosing this range by hand requires
inspecting the simulation output and estimating where the protonation fraction
crosses 50/50.

This notebook demonstrates how `auto_linspace` / `estimate_from_simulation` /
`estimate_from_results` automate that step.

## Why the centre matters

The equilibrium offset (50/50 protonation point) is determined by the free energy
difference between states:

$$\text{EIR}_{\text{eq}} = \text{EIR}_{\text{actual}} + \Delta F$$

where $\Delta F$ is read from dfmult's `df.out`.  The scan range is centred on
this point so that the reweighted sigmoid is fully captured.

## Three methods (all additive, no changes to existing workflow)

| Method | When to use | Requires |
|---|---|---|
| `estimate_from_results` | After post-processing | `results.out` |
| `estimate_from_simulation` | After ene_ana/dfmult | `df.out` + `e*r.dat` |
| `estimate_from_offset` | Fallback | `e*r.dat` only |
| `auto_linspace` | Any stage | Picks best available |

`batch_linspace` handles multiple runs at once and merges their ranges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from cpaeds.linspace_estimator import (
    auto_linspace,
    estimate_from_simulation,
    estimate_from_results,
    estimate_from_offset,
    batch_linspace,
    LinspaceResult,
)

## 1. Using the test fixture

The fixture directory `cpaeds/tests/test_data/reweighting/` contains all
required files: `e1r.dat`, `e2r.dat`, `df.out`, `eds_vr.dat`, and `results.out`.

In [ ]:
FIXTURE = str(Path("../cpaeds/tests/test_data/reweighting").resolve())

# Method 1: free-energy correction from df.out + e*r.dat
r_sim = estimate_from_simulation(FIXTURE, active_state=0)
print(f"Method: {r_sim.method}")
print(f"EIR_actual (from e1r.dat) = -220.0 kJ/mol")
print(f"ΔF        (from df.out)   = +2.5  kJ/mol")
print(f"=> Centre = EIR_actual + ΔF = {r_sim.center:.1f} kJ/mol")
print(f"=> Scan: {r_sim.lnspace[0]:.1f} to {r_sim.lnspace[-1]:.1f} ({r_sim.n_points} pts)")

In [ ]:
# Method 2: logistic fit to results.out
results_file = str(Path(FIXTURE) / "results.out")
r_res = estimate_from_results(results_file)
print(f"Method: {r_res.method}")
print(f"Logistic midpoint (50/50 crossing) = {r_res.center:.2f} kJ/mol")
print(f"Width derived from logistic steepness = {r_res.width:.1f} kJ/mol")

In [ ]:
# Method 3: auto_linspace picks the best available
r_auto = auto_linspace(FIXTURE, prefer="df")  # prefer df.out method
print(f"auto_linspace chose: '{r_auto.method}'")
print(f"Centre: {r_auto.center:.2f} kJ/mol")

## 2. Visualise the estimated scan range

In [ ]:
import pandas as pd
from cpaeds.algorithms import logistic_curve, log_fit

# Load and plot the results.out data
df = pd.read_csv(results_file, index_col=0)
offsets = df.filter(like='OFFSET').iloc[:, -1].values
fractions = df.filter(like='FRACTION').iloc[:, -1].values

fig, ax = plt.subplots(dpi=150, figsize=(8, 4))
ax.scatter(offsets, fractions, zorder=3, label='simulation data')

# Fit logistic to show the estimated midpoint
x_dense = np.linspace(offsets.min() - 20, offsets.max() + 20, 300)
popt = log_fit(offsets, fractions)
ax.plot(x_dense, logistic_curve(x_dense, *popt), 'r-', label='logistic fit')

# Mark the three estimates
for r, label, color in [
    (r_sim, f'ΔF correction ({r_sim.center:.1f})', 'green'),
    (r_res, f'logistic mid ({r_res.center:.1f})', 'red'),
]:
    ax.axvline(r.center, color=color, linestyle='--', linewidth=1.2, label=label)
    ax.axvspan(r.lnspace[0], r.lnspace[-1], alpha=0.08, color=color)

ax.set_xlabel('Offset (kJ/mol)')
ax.set_ylabel('Fraction')
ax.legend(fontsize=8)
ax.set_title('Estimated scan ranges overlaid on results')
plt.tight_layout()
plt.show()

## 3. Drop into a full reweighting workflow

In [ ]:
from cpaeds.reweighting import reweighting_constpH, parallel_reweight

# --- Step 1: get the linspace automatically ---
est = auto_linspace(
    path=FIXTURE,      # your ene_ana directory
    active_state=0,    # state whose offset is scanned
    n_points=101,
    width=50.0,        # scan ±25 kJ/mol around the equilibrium point
    prefer="df",       # use ΔF correction (works before full post-processing)
)
print(f"Scan: {est.lnspace[0]:.1f} → {est.lnspace[-1]:.1f} kJ/mol")
print(f"Centre: {est.center:.2f} kJ/mol  (method: {est.method})")

# --- Step 2: run reweighting over the estimated range ---
# (here using the fixture directly; in practice use parallel_reweight)
scan_results = []
for eir in est.lnspace:
    r = reweighting_constpH(
        cutoff=-400.0,
        temp=300.0,
        stepsize=0.5,
        itime=0.0,
        group=[["1"], ["0"]],
        path=FIXTURE,
        H1_eds=True,
        art_eir=[eir, 0.0],
    )
    scan_results.append(r)

A_fracs = np.array([r.A_frac for r in scan_results])

# --- Step 3: plot ---
fig, ax = plt.subplots(dpi=150, figsize=(7, 4))
ax.plot(est.lnspace, A_fracs, 'o-', ms=3)
ax.axvline(est.center, color='gray', linestyle='--', label=f'centre = {est.center:.1f}')
ax.set_xlabel('Artificial EIR (kJ/mol)')
ax.set_ylabel('Reweighted A fraction')
ax.set_title(f'Reweighted scan (auto-centred, method: {est.method})')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 4. Multiple runs: batch_linspace

When you have several statistical runs, their equilibrium offsets may differ slightly
(due to different sampling).  `batch_linspace` estimates the centre for each run
independently and returns a merged array that covers all of them.

In [ ]:
# Simulate three runs with the same fixture (in practice: different paths)
run_paths = [FIXTURE, FIXTURE, FIXTURE]

merged_lnspace, individual = batch_linspace(
    run_paths,
    active_state=0,
    n_points=101,
    width=50.0,
    prefer="df",
)

print("Individual centres (one per run):")
for i, r in enumerate(individual):
    print(f"  run {i+1}: {r.center:.2f} kJ/mol  ({r.method})")

print(f"\nMerged range: {merged_lnspace[0]:.1f} → {merged_lnspace[-1]:.1f} kJ/mol")
print(f"({len(merged_lnspace)} points covering all runs)")

## 5. Real-system usage (template)

Replace the paths below with your actual system:

```python
from cpaeds.linspace_estimator import auto_linspace, batch_linspace
from cpaeds.reweighting import parallel_reweight

# Single run
est = auto_linspace(
    path="/data/ASPD_trans_aq/ASPD_trans_aq_1_21/ene_ana",
    active_state=0,    # protonated state index
    n_points=101,
    width=50.0,        # standard 50 kJ/mol range
    prefer="df",       # use ΔF correction
)
print(f"EIR_eq = {est.center:.2f} kJ/mol")

# Multiple runs: merge ranges across all statistical repeats
run_dirs = [
    f"/data/ASPD_trans_aq/ASPD_trans_aq_1_{run}/ene_ana"
    for run in [5, 7, 8, 9]
]
merged, individuals = batch_linspace(run_dirs, active_state=0)

# Pass to parallel_reweight
results = parallel_reweight(
    runs=[5, 7, 8, 9],
    lnspace=merged,
    nthreads=14,
    path_template="/data/ASPD_trans_aq/ASPD_trans_aq_1_{run}/ene_ana",
    cutoff=-400.0,
    temp=300.0,
    stepsize=0.5,
    itime=4000.0,
    group=[["1", "2", "3", "4"], ["0"]],
    H1_eds=True,
)
```